In [1]:
# import sys
# !{sys.executable} -m pip install --upgrade numpy==1.21.6

In [2]:

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [3]:
df = pd.read_pickle('data/df.pkl')
df = df.drop('geometry', axis = 1)
columns = df.columns

In [4]:
df

,year,plot_id,ndvi_peak,ndvi_peak_doy,ndvi_min,ndvi_integral,ndvi_sos,ndvi_eos,ndvi_los,ndvi_greenup_slope,...,aspect_max_cos,aspect_max_sin,aspect_mean_cos,aspect_mean_sin,slope_x,slope_y,slope_squared,slope_log,local_relief,total_relief_log
0,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
1,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
2,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
3,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
4,2016,1,0.272585,276,0.0,86.618218,43.0,366.0,323.0,0.000272,...,-0.770268,-0.637720,-0.983934,0.178534,1.277051,-7.038067,51.165251,2.098385,11.058680,3.116342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161396,2023,38,0.308560,118,0.0,63.349538,47.0,163.0,116.0,0.001523,...,1.000000,-0.000496,-0.413871,0.910336,2.606121,-1.184835,8.195699,1.351396,1.472255,1.451969
161397,2023,38,0.308560,118,0.0,63.349538,47.0,163.0,116.0,0.001523,...,1.000000,-0.000496,-0.413871,0.910336,2.606121,-1.184835,8.195699,1.351396,1.472255,1.451969
161398,2023,38,0.308560,118,0.0,63.349538,47.0,163.0,116.0,0.001523,...,1.000000,-0.000496,-0.413871,0.910336,2.606121,-1.184835,8.195699,1.351396,1.472255,1.451969
161399,2023,38,0.308560,118,0.0,63.349538,47.0,163.0,116.0,0.001523,...,1.000000,-0.000496,-0.413871,0.910336,2.606121,-1.184835,8.195699,1.351396,1.472255,1.451969


In [5]:
# NDVI LSTM Notebook Starter


# 2️⃣ Load data
# Replace with your actual file path
# df = pd.read_csv('ndvi_weather_data.csv')

# 3️⃣ Define early/mid season days
early_days = list(range(0, 100))   # e.g., day 0-99
mid_days   = list(range(100, 200)) # e.g., day 100-199
timesteps = len(early_days + mid_days)

In [6]:
# 4️⃣ Prepare NDVI sequences
ndvi_cols = list(range(0, 366))  # daily NDVI columns
ndvi_daily = df[ndvi_cols].values

In [7]:
# 5️⃣ Prepare weather sequences (repeat each feature for daily alignment if necessary)
weather_features = ['tmean', 'tmin', 'tmax', 'ppt', 'vpdmax', 'vpdmin']
weather_daily = df[weather_features].values  # shape: (num_samples, num_weather_features)
# If weather is already daily-aligned, skip reshaping

In [8]:
# 6️⃣ Build per-sample sequences
X_seq = []
for i in range(df.shape[0]):  # for each sample/plot
    # NDVI early+mid season (reshape to [timesteps, 1])
    ndvi_seq = ndvi_daily[i, early_days + mid_days].reshape(-1, 1)
    
    # Weather features: repeat across timesteps or slice if daily weather
    # Here we broadcast daily features to match timesteps
    weather_seq = np.tile(weather_daily[i], (timesteps, 1))
    
    # Concatenate NDVI + weather along features axis
    X_seq.append(np.hstack([ndvi_seq, weather_seq]))

In [9]:
X_lstm = np.array(X_seq)  # shape: (num_samples, timesteps, num_features)
num_samples, timesteps, num_features = X_lstm.shape
print("X_lstm shape:", X_lstm.shape)

X_lstm shape: (161401, 200, 7)


In [10]:
# 7️⃣ Prepare target: late-season NDVI (days 200-365)
y_seq = ndvi_daily[:, 200:366]  # shape: (num_samples, 166)

In [11]:
# 8️⃣ Normalize X (recommended)
scaler = MinMaxScaler()
X_flat = X_lstm.reshape(-1, num_features)  # flatten for scaling
X_scaled = scaler.fit_transform(X_flat)
X_lstm = X_scaled.reshape(num_samples, timesteps, num_features)

In [12]:
# 9️⃣ Split train/test
X_train, X_test, y_train, y_test = train_test_split(X_lstm, y_seq, test_size=0.2, random_state=42)

In [13]:
# 10️⃣ Build LSTM model
model = Sequential()
model.add(LSTM(64, input_shape=(timesteps, num_features), return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(y_train.shape[1]))  # predict full late-season NDVI sequence

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 64)                18432     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense (Dense)               (None, 166)               10790     
                                                                 
Total params: 29,222
Trainable params: 29,222
Non-trainable params: 0
_________________________________________________________________


In [14]:
from tensorflow.keras.callbacks import EarlyStopping

# 1️⃣ Define early stopping
early_stop = EarlyStopping(
    monitor='val_loss',   # can also use 'val_mae'
    patience=10,          # stop if no improvement after 10 epochs
    restore_best_weights=True,  # roll back to the best model
    verbose=1
)

In [15]:
# 2️⃣ Train model with early stopping
history = model.fit(
    X_train, 
    y_train, 
    epochs=100,          # you can set a high number since early stopping will cut it
    batch_size=32, 
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/100
3632/3632 [==============================] - 278s 76ms/step - loss: 0.0029 - mae: 0.0393 - val_loss: 8.2491e-04 - val_mae: 0.0215
Epoch 2/100
 798/3632 [=====>........................] - ETA: 4:30 - loss: 9.2658e-04 - mae: 0.0230

KeyboardInterrupt: 

In [ ]:
# 11️⃣ Train model
# history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1)